# Adım 8 — Biyolojik yorum (keşifsel)

**Kod sürümü 08-1.1.** Bu not defteri **hipotez testi değildir**. Kilitli plan (v1.0), dondurulmuş model ve 4b–7. adımların sonuçları değiştirilmez. Sağkalım verisi **hiç okunmaz**; yeni bir sağkalım modeli kurulmaz. Test ve dış kohortların ifade dosyaları açılmaz. Bütün analizler yalnızca GSE181063 **eğitim** setinin birincil popülasyonunda (373 hasta) yapılır.

Sıra:
1. Kurulum ve önceden belirlenmiş analiz ayarları (analiz başlamadan dosyaya yazılır)
2. Bütünlük kontrolleri: plan kilidi, dondurulmuş model, 4b–7 manifestoları (geçmezse not defteri durur)
3. Eğitim verisinin yüklenmesi ve modelin eğitim verisinden birebir yeniden üretilmesi
4. 44 genin tablosu (katsayı yönü, bootstrap seçim sıklığı, güncel GPL570 sembolleri)
5. Yolak zenginleştirme: (a) aşırı temsil (ORA), (b) skorla korelasyona dayalı gen seti testi
6. Bilinen DLBCL imzalarıyla ilişki (Lymph2Cx/COO, proliferasyon, stroma vekilleri, MSigDB'deki DLBCL imzaları)
7. Literatür notları (yalnızca doğrulanmış kaynaklar)
8. Özet, manifesto, çalıştırma kaydı

> Gen seçimi 4b'de **kararsızdı** (44 genin yalnızca 4'ü bootstrap örneklemlerinin ≥%50'sinde seçildi). Tek tek genlere biyolojik anlam yüklemek risklidir; yorumun ağırlığı skorun bütününde ve yolak düzeyindedir.

## 1. Kurulum

Drive'ı bağlar, kütüphaneleri yükler, çıktı klasörlerini (`results/08`, `figures/08`, `data/external/msigdb_v2026.1.Hs`) oluşturur. Ek paket kurulumu gerekmez.

In [ ]:
# Step 8 - exploratory biological interpretation. Training data only; no survival data is read.
import os, sys, json, hashlib, re, time, platform, datetime, urllib.request, warnings

CODE_VERSION = '08-1.1'
PROJECT = os.environ.get('DLBCL_PROJECT', '/content/drive/MyDrive/dlbcl-prognosis')
if PROJECT.startswith('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        raise RuntimeError('Not running in Colab: set the DLBCL_PROJECT environment variable to the project folder.')

import numpy as np
import pandas as pd
import scipy
from scipy import stats, sparse
import matplotlib
import matplotlib.pyplot as plt

def P(*parts):
    return os.path.join(PROJECT, *parts)

RES, FIG = P('results', '08'), P('figures', '08')
MSIGDB_RELEASE = '2026.1.Hs'
GS_DIR = P('data', 'external', f'msigdb_v{MSIGDB_RELEASE}')
for d in (RES, FIG, GS_DIR):
    os.makedirs(d, exist_ok=True)
RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
print('Project:', PROJECT)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scipy', scipy.__version__)

## 2. Beklenen değerler ve önceden belirlenmiş analiz ayarları

İlk bölüm önceki adımlardan bilinen değerlerdir (özetler, hasta sayıları, COO dağılımı, bootstrap sıklıkları); bunlardan herhangi biri tutmazsa not defteri durur. İkinci bölüm bu adımın bütün analiz seçimleridir. Ayarlar sonuçlar görülmeden `analysis_config_08.json` dosyasına yazılır ve özeti manifestoya girer; böylece sonuç görüldükten sonra ayar değiştirilirse iz kalır.

**Arka plan (gen havuzu) seçimi:** model, final uydurmada eğitim verisinden seçilen **en değişken 5.000 gen** içinden seçildi; bu havuzun dışındaki bir genin modele girme şansı sıfırdı. Bu yüzden birincil arka plan bu 5.000 gendir. 13.747 ortak gen arka plan olarak kullanılırsa, yüksek varyanslı genlerin zaten zengin olduğu yolaklar (ör. bağışıklık/stroma) yapay olarak "zenginleşmiş" görünür. 13.747 gen yalnızca duyarlılık analizi olarak raporlanır.

In [ ]:
EXPECTED = {
    'plan_lock_sha256': '8d87cf887e2db4770ae3273755d4901cd0d5a2a010fdc729159e007aa129bc7d',
    'model_sha256': '661e2b80be774252a7fd613116ecc06673a8617ed8a2230bdae56170e3908403',
    'manifest_06_sha256_prefix': '0899a7594a1b',
    'n_model_genes': 44,
    'lambda': 0.086259,
    'training_lp_cutoff': -0.0048,
    'n_train': 380,
    'n_primary': 373,
    'n_qc_excluded': 7,
    'coo_counts_primary': {'GCB': 185, 'ABC': 104, 'UNC': 61, 'MHG': 23},
    'ipi_missing_primary': 92,
    'n_common_genes': 13747,
    'top_n_variance': 5000,
    'bootstrap_freq_check': {'PXDNL': 0.84, 'JAML': 0.68, 'PLAC8L1': 0.60, 'AGBL4': 0.50},
}
TEST_MODE = bool(os.environ.get('DLBCL_TEST_EXPECTED'))
if TEST_MODE:   # used only for offline testing with synthetic data; never set in Colab
    EXPECTED.update(json.load(open(os.environ['DLBCL_TEST_EXPECTED'])))
    print('!!! TEST MODE: expected values overridden from', os.environ['DLBCL_TEST_EXPECTED'])

SEED = 20261815        # step-8 master seed: permutations SEED + 1, bootstrap SEED + 2
MSIGDB_BASE = f'https://data.broadinstitute.org/gsea-msigdb/msigdb/release/{MSIGDB_RELEASE}/'
GMT_FILES = {
    'HALLMARK': f'h.all.v{MSIGDB_RELEASE}.entrez.gmt',
    'REACTOME': f'c2.cp.reactome.v{MSIGDB_RELEASE}.entrez.gmt',
    'GO_BP':    f'c5.go.bp.v{MSIGDB_RELEASE}.entrez.gmt',
    'C2_CGP':   f'c2.cgp.v{MSIGDB_RELEASE}.entrez.gmt',
}
CONFIG = {
    'code_version': CODE_VERSION,
    'population': 'GSE181063 training set, primary population (sex_mismatch_confident excluded); survival data never read',
    'score': 'frozen 44-gene linear predictor (model_04.json) on training expression, apparent (in-sample)',
    'msigdb_release': MSIGDB_RELEASE, 'msigdb_files': GMT_FILES, 'msigdb_base_url': MSIGDB_BASE,
    'background_primary': 'top-5000 variance genes of the final 4b fit (the pool the model was selected from)',
    'background_sensitivity': 'all 13,747 common genes',
    'ora': {'test': 'one-sided hypergeometric (over-representation)', 'gene_lists': ['all_44', 'risk_up (coef > 0)', 'risk_down (coef < 0)'],
            'collections': ['HALLMARK', 'REACTOME', 'GO_BP'], 'set_size_in_background': [10, 500],
            'multiple_testing': 'Benjamini-Hochberg within each gene list x collection x background'},
    'rank_test': {'statistic': 'mean Spearman rho(gene, LP) in set minus mean over universe',
                  'universe': 'primary background minus the 44 model genes (removes circularity)',
                  'null': 'patient-label permutation of LP (preserves gene-gene correlation)', 'n_perm': 2000,
                  'collections': ['HALLMARK', 'REACTOME', 'GO_BP'], 'set_size_in_universe': [10, 500],
                  'second_test': 'PAGE-type gene-sampling z (Kim & Volsky 2005), normal approximation with finite-population correction',
                  'highlight_rule': 'robust = BH FDR < 0.05 in BOTH the permutation and the gene-sampling test',
                  'multiple_testing': 'Benjamini-Hochberg within collection, separately for each test'},
    'signatures': {
        'score': 'mean of per-gene z-scores (primary population, ddof 1) over genes found among the 13,747 common genes',
        'lymph2cx': 'Scott et al. Blood 2014; COO score = mean z(ABC genes) - mean z(GCB genes)',
        'curated_hallmark_panel': ['HALLMARK_E2F_TARGETS', 'HALLMARK_G2M_CHECKPOINT', 'HALLMARK_MYC_TARGETS_V1',
                                   'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION', 'HALLMARK_ANGIOGENESIS',
                                   'HALLMARK_INTERFERON_GAMMA_RESPONSE', 'HALLMARK_INFLAMMATORY_RESPONSE',
                                   'HALLMARK_TNFA_SIGNALING_VIA_NFKB', 'HALLMARK_IL6_JAK_STAT3_SIGNALING'],
        'proxies': {'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION': 'ECM / stromal-1 proxy (not the Lenz 2008 gene list)',
                    'HALLMARK_ANGIOGENESIS': 'vascular / stromal-2 proxy (not the Lenz 2008 gene list)'},
        'c2_cgp_regex': r'(DLBCL|DIFFUSE_LARGE|LARGE_B_CELL|LYMPHOMA|^LENZ_|^ROSENWALD_|^MONTI_|^SHIPP_|^ALIZADEH_|^SHAFFER_|^WRIGHT_)',
        'min_genes_found': 10, 'association': 'Spearman rho with LP, 1000 patient bootstraps (percentile CI)',
        'model_gene_overlap': 'rho also reported with the 44 model genes removed from each signature',
        'multiple_testing': 'Benjamini-Hochberg within curated panel and within the C2-CGP group separately',
        'regression': 'OLS of LP on standardized signature scores (model genes removed), descriptive R^2 only'},
    'fdr_threshold': 0.05, 'seed': SEED,
}
cfg_path = os.path.join(RES, 'analysis_config_08.json')
with open(cfg_path, 'w') as f:
    json.dump(CONFIG, f, indent=2, sort_keys=True)
print('Analysis configuration written before any analysis:', cfg_path)

## 3. Yardımcı fonksiyonlar

SHA-256 özeti, manifestolardaki dosya özetlerini (yapıları bilinmediği için) her düzende bulan bir tarayıcı, tablo sütunu bulucular, Benjamini–Hochberg düzeltmesi ve GMT okuyucu.

In [ ]:
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

HEX64 = re.compile(r'^(?:sha256:)?([0-9a-fA-F]{64})$')
FILE_RE = re.compile(r'\.(csv|tsv|json|jsonl|parquet|md|txt|png|pdf|py|ipynb|gz|npz|npy|pkl|joblib|gmt)$', re.I)
NAME_KEYS = ('path', 'file', 'filename', 'name', 'relpath', 'relative_path', 'file_path')
HASH_KEYS = ('sha256', 'sha', 'hash', 'sha256_hex', 'digest')

def _hash_in(d):
    for k in HASH_KEYS:
        v = d.get(k)
        if isinstance(v, str) and HEX64.match(v):
            return HEX64.match(v).group(1).lower()
    return None

def iter_file_hashes(obj):
    # Yield (recorded_name, sha256) for every file hash recorded in a JSON object, whatever its layout.
    if isinstance(obj, dict):
        name = next((obj[k] for k in NAME_KEYS if isinstance(obj.get(k), str) and FILE_RE.search(obj[k])), None)
        h = _hash_in(obj)
        if name and h:
            yield name, h
        for k, v in obj.items():
            if isinstance(k, str) and FILE_RE.search(k):
                if isinstance(v, str) and HEX64.match(v):
                    yield k, HEX64.match(v).group(1).lower()
                elif isinstance(v, dict) and _hash_in(v):
                    yield k, _hash_in(v)
            yield from iter_file_hashes(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_file_hashes(v)

def all_strings(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield str(k)
            yield from all_strings(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from all_strings(v)
    elif isinstance(obj, str):
        yield obj

SKIP_DIRS = {'.ipynb_checkpoints', '08'}
_index = None
def file_index():
    global _index
    if _index is None:
        _index = {}
        for root, dirs, files in os.walk(PROJECT):
            dirs[:] = [d for d in dirs if d not in SKIP_DIRS and not d.startswith('msigdb_v')]
            for fn in files:
                _index.setdefault(fn, []).append(os.path.join(root, fn))
    return _index

def resolve(name):
    # Map a recorded file name to a path under PROJECT: relative path, project-anchored absolute path, or unique basename.
    n = name.replace('\\', '/')
    if 'dlbcl-prognosis/' in n:
        n = n.split('dlbcl-prognosis/', 1)[1]
    cand = n if os.path.isabs(n) else P(n)
    if os.path.isfile(cand):
        return cand, 'path'
    hits = file_index().get(os.path.basename(n), [])
    if len(hits) == 1:
        return hits[0], 'basename'
    return None, ('ambiguous' if hits else 'missing')

_hash_cache = {}
def cached_sha(path):
    if path not in _hash_cache:
        _hash_cache[path] = sha256(path)
    return _hash_cache[path]

MAX_HASH_BYTES = 600 * 2**20
def verify_records(label, obj):
    # Verify every resolvable file hash recorded in obj; notebooks (.ipynb) are only warned about,
    # because Colab re-saves notebooks after a run.
    out = {'label': label, 'records': 0, 'verified': 0, 'mismatch': [], 'notebook_mismatch': [],
           'unresolved': 0, 'skipped_large': 0, 'verified_paths': {}}
    seen = set()
    for name, h in iter_file_hashes(obj):
        if (name, h) in seen:
            continue
        seen.add((name, h))
        out['records'] += 1
        path, how = resolve(name)
        if path is None:
            out['unresolved'] += 1
            continue
        if os.path.getsize(path) > MAX_HASH_BYTES:
            out['skipped_large'] += 1
            continue
        actual = cached_sha(path)
        if actual == h:
            out['verified'] += 1
            out['verified_paths'][os.path.relpath(path, PROJECT)] = h
        elif path.endswith('.ipynb'):
            out['notebook_mismatch'].append(os.path.relpath(path, PROJECT))
        else:
            out['mismatch'].append({'file': os.path.relpath(path, PROJECT), 'recorded': h[:12], 'actual': actual[:12]})
    return out

def find_col(columns, candidates):
    low = {c.lower(): c for c in columns}
    for c in candidates:
        if c.lower() in low:
            return low[c.lower()]
    return None

def norm_entrez(v):
    try:
        return str(int(float(v)))
    except (TypeError, ValueError):
        return str(v).strip()

def to_bool(s):
    return s.map(lambda v: str(v).strip().lower() in ('1', '1.0', 'true', 't', 'yes', 'y'))

def bh(p):
    p = np.asarray(p, dtype=float)
    out = np.full(p.shape, np.nan)
    ok = ~np.isnan(p)
    if ok.sum() == 0:
        return out
    q = p[ok]
    order = np.argsort(q)
    ranked = q[order] * len(q) / np.arange(1, len(q) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    res = np.empty_like(q)
    res[order] = np.minimum(ranked, 1.0)
    out[ok] = res
    return out

def read_gmt(path):
    sets = {}
    with open(path) as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 3:
                continue
            sets[parts[0]] = {norm_entrez(g) for g in parts[2:] if g.strip()}
    if not sets:
        raise ValueError(f'No gene sets parsed from {path}')
    return sets

def save_fig(fig, name):
    path = os.path.join(FIG, name)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return path

CHECKS = []
def check(name, ok, detail=''):
    CHECKS.append({'check': name, 'ok': bool(ok), 'detail': str(detail)})
    print(('PASS ' if ok else 'FAIL ') + name + (f' - {detail}' if detail else ''))
    if not ok:
        raise RuntimeError(f'Check failed: {name}. {detail}  Stopping: nothing has been analysed.')
print('Helpers ready.')

## 4. Bütünlük kontrolleri (kapı)

- Plan kilidinin SHA-256'sı beklenen değerle aynı mı; kilidin kaydettiği her dosya (iki plan dosyası ve girdi verileri) hâlâ aynı mı.
- `model_04.json` özeti beklenen değer mi ve `manifest_04b.json` bu özeti içeriyor mu.
- 05, 06, 07 manifestoları var mı; kaydettikleri her dosya hâlâ aynı mı; `manifest_06.json` özeti beklenen önekle başlıyor mu.
- Bu adımın okuyacağı her veri dosyası en az bir kayıtla (kilit ya da manifesto) doğrulanıyor mu.

Herhangi bir uyuşmazlıkta not defteri durur ve hiçbir analiz yapılmaz. Colab not defterleri (.ipynb) çalıştırmadan sonra yeniden kaydedildiği için onların farkı yalnızca uyarıdır.

In [ ]:
PATHS = {
    'lock': P('results', '04', 'plan_lock_04.json'),
    'amendments': P('results', '04', 'plan_amendments.md'),
    'model': P('results', '04', 'model_04.json'),
    'manifest_04b': P('results', '04', 'manifest_04b.json'),
    'manifest_05': P('results', '05', 'manifest_05.json'),
    'manifest_06': P('results', '06', 'manifest_06.json'),
    'manifest_07': P('results', '07', 'manifest_07.json'),
    'first_results_07': P('results', '07', 'first_results_07.json'),
    'expr_train': P('data', 'processed', 'expr_GSE181063_train.parquet'),
    'gene_table': P('data', 'processed', 'gene_table_common.csv'),
    'qc': P('data', 'processed', 'qc_samples.csv'),
    'clinical': P('data', 'clean', 'clinical_clean_all.csv'),
    'stability': P('results', '04', 'stability_04b.csv'),
    'variance_filter': P('results', '04', 'variance_filter_genes_04b.csv'),
}
for k, p in PATHS.items():
    check(f'file exists: {k}', os.path.isfile(p), os.path.relpath(p, PROJECT))

lock_sha = sha256(PATHS['lock'])
check('plan lock SHA-256', lock_sha == EXPECTED['plan_lock_sha256'], lock_sha[:12])
model_sha = sha256(PATHS['model'])
check('frozen model SHA-256', model_sha == EXPECTED['model_sha256'], model_sha[:12])

manifests = {k: json.load(open(PATHS[k])) for k in ('lock', 'manifest_04b', 'manifest_05', 'manifest_06', 'manifest_07')}
check('manifest_04b records the frozen model hash', any(model_sha in s.lower() for s in all_strings(manifests['manifest_04b'])))
m06 = sha256(PATHS['manifest_06'])
check('manifest_06 SHA-256 prefix', m06.startswith(EXPECTED['manifest_06_sha256_prefix']), m06[:12])

def skeleton(obj, depth=0, maxdepth=3):
    # Structure only (keys, types, value lengths) - printed when a check fails, safe to share.
    pad = '  ' * depth
    if isinstance(obj, dict):
        for k, v in list(obj.items())[:25]:
            t = type(v).__name__ + (f'[{len(v)}]' if isinstance(v, (list, dict, str)) else '')
            print(f'{pad}{k}: {t}')
            if depth < maxdepth and isinstance(v, (dict, list)):
                skeleton(v, depth + 1, maxdepth)
    elif isinstance(obj, list) and obj:
        print(f'{pad}[0]: {type(obj[0]).__name__}')
        if depth < maxdepth and isinstance(obj[0], (dict, list)):
            skeleton(obj[0], depth + 1, maxdepth)

HEXES = {k: {HEX64.match(s).group(1).lower() for s in all_strings(obj) if HEX64.match(s)} for k, obj in manifests.items()}

# (1) Name-based verification where file names are recorded (any mismatch is fatal).
verification = {}
for k, obj in manifests.items():
    r = verify_records(k, obj)
    verification[k] = r
    print(f"  {k:13s} hashes {len(HEXES[k]):4d} | named records {r['records']:4d} | verified by name {r['verified']:4d}"
          f" | unresolved {r['unresolved']:3d} | notebook diffs {len(r['notebook_mismatch'])}")
    if r['notebook_mismatch']:
        print('    warning: notebook re-saved since recording (not fatal):', r['notebook_mismatch'])
    check(f'{k}: no changed files among named records', not r['mismatch'], r['mismatch'][:5])

# (2) Layout-independent verification: the current SHA-256 of each file must appear somewhere in the record.
def must_appear(file_key, where):
    h = cached_sha(PATHS[file_key]) if file_key in PATHS else cached_sha(file_key)
    ok = any(h in HEXES[w] for w in where)
    if not ok:
        print(f'\nDiagnostic - structure of {where} (keys and types only, no data):')
        for w in where:
            print(f'--- {w}'); skeleton(manifests[w])
    return ok, h

PATHS['plan_md'] = P('results', '04', 'analysis_plan_04.md')
PATHS['plan_json'] = P('results', '04', 'analysis_plan_04.json')
for k in ('plan_md', 'plan_json', 'expr_train', 'gene_table', 'qc', 'clinical'):
    check(f'file exists: {k}', os.path.isfile(PATHS[k]))
    ok, h = must_appear(k, ['lock'])
    check(f'current SHA-256 of {k} recorded in the plan lock', ok, h[:12])
ok, h = must_appear('model', ['manifest_04b'])
check('frozen model hash recorded in manifest_04b', ok, h[:12])
for m in ('manifest_05', 'manifest_06'):
    check(f'frozen model hash recorded in {m}', model_sha in HEXES[m])
print('manifest_07 records the model hash:', model_sha in HEXES['manifest_07'], '(informational)')
for k in ('stability', 'variance_filter'):
    h = cached_sha(PATHS[k])
    if any(h in HEXES[w] for w in HEXES):
        print(f'PASS current SHA-256 of {k} recorded in a manifest')
    else:
        print(f'note: {k} hash not found in any manifest; it is cross-checked against known values below')

amend = open(PATHS['amendments']).read()
print('plan_amendments.md size:', len(amend), 'characters | hash recorded in lock:', cached_sha(PATHS['amendments']) in HEXES['lock'])
DATA_READ = sorted({os.path.relpath(PATHS[k], PROJECT) for k in
                    ('model', 'expr_train', 'gene_table', 'qc', 'clinical', 'stability', 'variance_filter')})
print('\nGate passed. Data files this notebook reads:', DATA_READ)
print('Test-set and external-cohort expression files are never opened.')

## 5. Eğitim verisinin yüklenmesi ve modelin yeniden üretilmesi

- Klinik tablodan **yalnızca** örnek kimliği, COO ve IPI sütunları okunur; sağkalım sütunları belleğe hiç girmez.
- Birincil popülasyon: 380 eğitim örneği − 7 `sex_mismatch_confident` = 373 (sample_id sırasına göre).
- Kontroller: hasta sayısı, COO dağılımı, IPI eksikliği; modeldeki eğitim ortalama/SD'lerinin eğitim verisinden birebir yeniden hesaplanması; eğitim LP medyanının kayıtlı eşiğe eşitliği; 5.000 genlik varyans filtresinin yeniden hesaplanıp 4b dosyasıyla aynı olması.

In [ ]:
model = json.load(open(PATHS['model']))
genes = pd.DataFrame(model['genes'])
genes['entrez'] = genes['entrez'].map(norm_entrez)
check('model has expected number of genes', len(genes) == EXPECTED['n_model_genes'], len(genes))
lam = (model.get('penalty') or {}).get('lambda')
check('model lambda', lam is not None and abs(float(lam) - EXPECTED['lambda']) < 1e-6, lam)

X = pd.read_parquet(PATHS['expr_train'])
X.index = [norm_entrez(v) for v in X.index]
check('training expression: genes x samples', X.shape == (EXPECTED['n_common_genes'], EXPECTED['n_train']), X.shape)
check('gene identifiers unique', X.index.is_unique)
train_ids = [str(c) for c in X.columns]
X.columns = train_ids

def id_column(path, ids):
    head = pd.read_csv(path, nrows=0).columns
    cands = [c for c in head if re.search(r'sample|gsm|geo|accession|^id$|array', c, re.I)] or list(head)
    best, best_n = None, -1
    for c in cands:
        vals = set(pd.read_csv(path, usecols=[c], dtype=str)[c].dropna().str.strip())
        n = len(vals & set(ids))
        if n > best_n:
            best, best_n = c, n
    return best, best_n, head

qc_id, n_hit, qc_cols = id_column(PATHS['qc'], train_ids)
c_sex = find_col(qc_cols, ['sex_mismatch_confident'])
c_out = find_col(qc_cols, ['corr_outlier'])
check('QC table columns found', c_sex is not None and c_out is not None and n_hit == len(train_ids), (qc_id, n_hit))
qc = pd.read_csv(PATHS['qc'], usecols=[qc_id, c_sex, c_out], dtype=str)
qc[qc_id] = qc[qc_id].str.strip()
qc = qc[qc[qc_id].isin(train_ids)].drop_duplicates(qc_id).set_index(qc_id)
excluded = sorted(qc.index[to_bool(qc[c_sex])])
check('QC exclusions in training set', len(excluded) == EXPECTED['n_qc_excluded'], len(excluded))
primary = sorted(set(train_ids) - set(excluded))
check('primary population size', len(primary) == EXPECTED['n_primary'], len(primary))

cl_id, n_hit, cl_cols = id_column(PATHS['clinical'], train_ids)
c_coo, c_ipi = find_col(cl_cols, ['coo', 'cell_of_origin']), find_col(cl_cols, ['ipi', 'ipi_score'])
check('clinical columns found (id, coo, ipi only)', c_coo is not None and c_ipi is not None and n_hit >= len(primary),
      (cl_id, c_coo, c_ipi, n_hit))
clin = pd.read_csv(PATHS['clinical'], usecols=[cl_id, c_coo, c_ipi], dtype={cl_id: str})   # survival columns never loaded
clin[cl_id] = clin[cl_id].str.strip()
clin = clin[clin[cl_id].isin(primary)].drop_duplicates(cl_id).set_index(cl_id).reindex(primary)
COO_MAP = {'ABC': 'ABC', 'GCB': 'GCB', 'MHG': 'MHG', 'UNC': 'UNC', 'UNCLASSIFIED': 'UNC', 'U': 'UNC', 'TYPE III': 'UNC'}
clin['coo'] = clin[c_coo].astype(str).str.strip().str.upper().map(lambda v: COO_MAP.get(v, v))
clin['ipi'] = pd.to_numeric(clin[c_ipi], errors='coerce')
coo_counts = clin['coo'].value_counts().to_dict()
check('COO distribution (primary)', all(coo_counts.get(k, 0) == v for k, v in EXPECTED['coo_counts_primary'].items())
      and sum(coo_counts.values()) == len(primary), coo_counts)
check('IPI missing (primary)', int(clin['ipi'].isna().sum()) == EXPECTED['ipi_missing_primary'], int(clin['ipi'].isna().sum()))

Xp = X[primary].T                     # samples x genes, primary population, sample_id order
gene_ids = list(Xp.columns)
gpos = {g: i for i, g in enumerate(gene_ids)}
A = Xp.to_numpy(dtype=float)
check('all model genes present in expression', genes['entrez'].isin(gpos).all(),
      genes.loc[~genes['entrez'].isin(gpos), 'entrez'].tolist())

mp = genes['entrez'].map(gpos).to_numpy()
mean_re, sd_re = A[:, mp].mean(axis=0), A[:, mp].std(axis=0, ddof=1)
tol = lambda a, b: np.all(np.abs(a - b) <= 1e-6 * np.maximum(1.0, np.abs(b)))
check('model train_mean reproduced from training data', tol(mean_re, genes['train_mean'].to_numpy(float)),
      float(np.max(np.abs(mean_re - genes['train_mean'].to_numpy(float)))))
check('model train_sd reproduced from training data', tol(sd_re, genes['train_sd'].to_numpy(float)),
      float(np.max(np.abs(sd_re - genes['train_sd'].to_numpy(float)))))
coef = genes['coef'].to_numpy(float)
LP = ((A[:, mp] - genes['train_mean'].to_numpy(float)) / genes['train_sd'].to_numpy(float)) @ coef
cut = float(model['risk_groups']['cutoff'])
check('training LP median equals frozen cutoff', abs(np.median(LP) - cut) <= 1e-9 + 1e-7 * abs(cut), (np.median(LP), cut))
check('frozen cutoff matches step-5 record', abs(cut - EXPECTED['training_lp_cutoff']) < 5e-5, cut)
LPs = (LP - LP.mean()) / LP.std(ddof=1)

var = A.var(axis=0, ddof=1)
top_idx = np.sort(np.argsort(-var, kind='stable')[:EXPECTED['top_n_variance']])
BG5000 = [gene_ids[i] for i in top_idx]
BG_ALL = list(gene_ids)
check('all 44 model genes inside the recomputed 5,000-gene pool', genes['entrez'].isin(set(BG5000)).all())

vf = pd.read_csv(PATHS['variance_filter'], dtype=str)
vf_set = None
for c in vf.columns:
    vals = vf[c].dropna().map(norm_entrez)
    if vals.isin(gpos).mean() > 0.99:
        flag = find_col(vf.columns, ['selected', 'in_filter', 'kept', 'in_top', 'top5000'])
        if len(vals) == EXPECTED['top_n_variance']:
            vf_set = set(vals)
        elif flag is not None:
            vf_set = set(vf.loc[to_bool(vf[flag]), c].map(norm_entrez))
        break
if vf_set is None:
    print('note: could not parse the 4b variance-filter file; using the recomputed pool (deterministic, ddof 1, stable sort)')
else:
    check('recomputed 5,000-gene pool identical to 4b file', vf_set == set(BG5000), len(vf_set ^ set(BG5000)))
print(f'\nPrimary population: {len(primary)} patients | LP mean {LP.mean():.4f}, SD {LP.std(ddof=1):.4f}, median {np.median(LP):.4f}')

## 6. 44 genin tablosu

Her gen için: katsayı (z-skor başına log HR, cezalı modelde), yön (**pozitif = yüksek ifade, yüksek risk**; negatif = yüksek ifade, düşük risk), 4b'nin 100 bootstrap'ındaki seçim sıklığı, güncel GPL570 (NCBI `GPL570.annot`) sembolü ve GPL14951 (SOFT) sembolü.

Okuma notu: katsayılar cezalı ve çok değişkenli bir modelden gelir; tek bir genin "bağımsız etkisi" değildir. Sıklık < %50 olan bir genin modelde olması büyük ölçüde rastlantıya bağlıdır.

In [ ]:
gt = pd.read_csv(PATHS['gene_table'], dtype=str)
gt['gene'] = gt['gene'].map(norm_entrez)
gt = gt.drop_duplicates('gene').set_index('gene')
c14 = find_col(gt.columns, ['symbol', 'symbol_GPL14951'])
c570 = find_col(gt.columns, ['symbol_GPL570'])
check('gene table symbol columns', c14 is not None and c570 is not None, list(gt.columns)[:8])
def sym(e):
    s570 = gt[c570].get(e) if e in gt.index else None
    s14 = gt[c14].get(e) if e in gt.index else None
    cur = s570 if isinstance(s570, str) and s570.strip() else (s14 if isinstance(s14, str) else e)
    return cur, (s14 if isinstance(s14, str) else '')
SYM = {e: sym(e)[0] for e in gene_ids}

ALIASES = {'JAML': {'JAML', 'AMICA1'}, 'RAB29': {'RAB29', 'RAB7L1'}, 'RAB7L1': {'RAB29', 'RAB7L1'}}
SYM2E = {}
for e in gt.index:
    for c in (c570, c14):
        s = gt.at[e, c]
        if isinstance(s, str) and s.strip():
            SYM2E.setdefault(s.strip().upper(), set()).add(e)
def entrez_for(symbol):
    hits = set()
    for s in ALIASES.get(symbol.upper(), {symbol.upper()}):
        hits |= SYM2E.get(s, set())
    hits = {h for h in hits if h in gpos}
    return sorted(hits)[0] if len(hits) == 1 else None

# --- bootstrap selection frequencies from 4b (layout detected, then cross-checked against known values)
st = pd.read_csv(PATHS['stability'], dtype=str)
model_set = set(genes['entrez'])
key_col, key_kind = None, None
for c in st.columns:
    if st[c].dropna().map(norm_entrez).isin(model_set).sum() >= 30:
        key_col, key_kind = c, 'entrez'; break
if key_col is None:
    model_syms = {SYM[e].upper() for e in model_set} | {sym(e)[1].upper() for e in model_set}
    for c in st.columns:
        if st[c].dropna().str.upper().isin(model_syms).sum() >= 30:
            key_col, key_kind = c, 'symbol'; break
check('stability table key column found', key_col is not None, list(st.columns))
st['_entrez'] = st[key_col].map(norm_entrez) if key_kind == 'entrez' else st[key_col].map(lambda s: entrez_for(str(s)))
known = {entrez_for(s): v for s, v in EXPECTED['bootstrap_freq_check'].items()}
check('known stable genes mapped to Entrez', None not in known, EXPECTED['bootstrap_freq_check'])
freq_col = None
for c in st.columns:
    if c in (key_col, '_entrez'):
        continue
    v = pd.to_numeric(st[c], errors='coerce')
    if v.notna().mean() < 0.9 or v.min() < 0 or v.max() > 100:
        continue
    f = v / 100.0 if v.max() > 1.0 else v
    fm = dict(zip(st['_entrez'], f))
    if all(e in fm and abs(fm[e] - val) <= 0.005 for e, val in known.items()):
        freq_col, FREQ = c, fm; break
check('bootstrap frequency column reproduces 4b values (PXDNL 84%, JAML 68%, PLAC8L1 60%, AGBL4 50%)',
      freq_col is not None, list(st.columns))

rows = []
for _, g in genes.iterrows():
    cur, s14 = sym(g['entrez'])
    fr = FREQ.get(g['entrez'], np.nan)
    rows.append({'entrez': g['entrez'], 'symbol_GPL570_current': cur, 'symbol_GPL14951': s14,
                 'symbol_differs': bool(s14) and s14 != cur, 'coef': g['coef'], 'hr_per_sd_penalized': np.exp(g['coef']),
                 'direction': 'higher expression -> higher risk' if g['coef'] > 0 else 'higher expression -> lower risk',
                 'bootstrap_freq': fr, 'stable_ge_50pct': bool(fr >= 0.5) if not np.isnan(fr) else False,
                 'train_mean': g['train_mean'], 'train_sd': g['train_sd']})
gene_tab = pd.DataFrame(rows)
missing_freq = gene_tab['bootstrap_freq'].isna().sum()
if missing_freq:
    print(f'note: {missing_freq} model genes absent from the stability table; treated as frequency 0')
    gene_tab['bootstrap_freq'] = gene_tab['bootstrap_freq'].fillna(0.0)
gene_tab['abs_coef_rank'] = gene_tab['coef'].abs().rank(ascending=False, method='first').astype(int)
gene_tab = gene_tab.sort_values(['bootstrap_freq', 'abs_coef_rank'], ascending=[False, True]).reset_index(drop=True)
gene_tab.to_csv(os.path.join(RES, 'model_genes_annotated_08.csv'), index=False)

n_up, n_down = int((gene_tab['coef'] > 0).sum()), int((gene_tab['coef'] < 0).sum())
print(f'Risk-up genes (coef > 0): {n_up} | risk-down genes (coef < 0): {n_down} | '
      f'stable (>= 50%): {int(gene_tab.stable_ge_50pct.sum())} | symbol differs between platforms: {int(gene_tab.symbol_differs.sum())}')
with pd.option_context('display.max_rows', 60, 'display.width', 200):
    display(gene_tab[['symbol_GPL570_current', 'symbol_GPL14951', 'entrez', 'coef', 'direction', 'bootstrap_freq', 'abs_coef_rank']])

fig, ax = plt.subplots(figsize=(7, 10))
gt_plot = gene_tab.sort_values('coef')
colors = ['#b2182b' if c > 0 else '#2166ac' for c in gt_plot['coef']]
ax.barh(range(len(gt_plot)), gt_plot['coef'], color=colors)
ax.set_yticks(range(len(gt_plot)))
ax.set_yticklabels([f"{s} ({f:.0%})" for s, f in zip(gt_plot['symbol_GPL570_current'], gt_plot['bootstrap_freq'])], fontsize=7)
ax.axvline(0, color='black', lw=0.6)
ax.set_xlabel('Penalized coefficient (log HR per SD of gene)')
ax.set_title('44-gene model: red = higher expression, higher risk; blue = lower risk\n(label: bootstrap selection frequency in 4b)', fontsize=9)
save_fig(fig, 'model_genes_coefficients_08.png')

## 7. MSigDB gen setleri

MSigDB v2026.1.Hs'den Entrez kimlikli GMT dosyaları (Hallmark, Reactome, GO Biyolojik Süreç, C2 CGP) indirilir ve Drive'a önbelleğe alınır; sonraki çalıştırmalar aynı dosyaları kullanır. Her dosyanın SHA-256'sı manifestoya yazılır. İndirme başarısız olursa hata mesajı dosyayı elle yükleme yolunu söyler.

Kaynak: Subramanian ve ark. 2005 (PNAS); Liberzon ve ark. 2015 (Cell Systems, Hallmark). MSigDB lisans koşullarına göre bu setler atıfla kullanılabilir.

In [ ]:
def get_gmt(fn):
    path = os.path.join(GS_DIR, fn)
    if not os.path.isfile(path):
        url = MSIGDB_BASE + fn
        print('downloading', url)
        try:
            with urllib.request.urlopen(url, timeout=120) as r, open(path + '.part', 'wb') as f:
                f.write(r.read())
            os.replace(path + '.part', path)
        except Exception as e:
            if os.path.exists(path + '.part'):
                os.remove(path + '.part')
            raise RuntimeError(f'Could not download {url} ({e}). Download it manually and upload it to {GS_DIR}, then re-run.')
    return path

GENESETS, GMT_RECORD = {}, {}
for key, fn in GMT_FILES.items():
    path = get_gmt(fn)
    GENESETS[key] = read_gmt(path)
    GMT_RECORD[fn] = {'sha256': sha256(path), 'url': MSIGDB_BASE + fn, 'n_sets': len(GENESETS[key])}
    print(f"{key:9s} {len(GENESETS[key]):6d} sets  sha256 {GMT_RECORD[fn]['sha256'][:12]}")
check('Hallmark collection has 50 sets', len(GENESETS['HALLMARK']) == 50, len(GENESETS['HALLMARK']))
cov = np.mean([g in set().union(*GENESETS['HALLMARK'].values()) for g in BG5000])
print(f'Fraction of the 5,000-gene background found in any Hallmark set: {cov:.2f}')

## 8a. Aşırı temsil analizi (ORA)

Soru: 44 gen (ve yönlere göre iki alt liste) belirli yolaklarda arka plana göre fazla mı? Test: tek yönlü hipergeometrik test; setler arka plana kısıtlanır, arka plandaki boyutu 10–500 olan setler test edilir; Benjamini–Hochberg (BH) düzeltmesi her liste × koleksiyon × arka plan için ayrı.

Beklenti (önceden): 44 genlik ve kararsız bir listeyle ORA'nın gücü çok düşüktür. Hiçbir setin FDR < 0,05 olmaması "biyoloji yok" demek değildir; bu yüzden 8b'deki skor tabanlı test ana yolak analizidir.

In [ ]:
def ora(query, background, gsets, lo=10, hi=500):
    bg = set(background); q = set(query) & bg
    N, n = len(bg), len(q)
    rows = []
    for name, gs in gsets.items():
        s = gs & bg; K = len(s)
        if K < lo or K > hi:
            continue
        hit = sorted(s & q); k = len(hit)
        rows.append({'gene_set': name, 'set_size_in_bg': K, 'overlap': k, 'expected': n * K / N,
                     'fold_enrichment': (k / (n * K / N)) if K else np.nan,
                     'p': float(stats.hypergeom.sf(k - 1, N, K, n)) if k > 0 else 1.0,
                     'overlap_genes': ';'.join(SYM.get(g, g) for g in hit)})
    df = pd.DataFrame(rows)
    df['fdr_bh'] = bh(df['p'].to_numpy()) if len(df) else []
    return df.sort_values('p').reset_index(drop=True)

LISTS = {'all_44': list(gene_tab['entrez']),
         'risk_up': list(gene_tab.loc[gene_tab.coef > 0, 'entrez']),
         'risk_down': list(gene_tab.loc[gene_tab.coef < 0, 'entrez'])}
ORA_ALL = []
for bg_name, bg in (('primary_top5000', BG5000), ('sensitivity_all13747', BG_ALL)):
    for lname, lst in LISTS.items():
        for coll in ('HALLMARK', 'REACTOME', 'GO_BP'):
            d = ora(lst, bg, GENESETS[coll])
            d.insert(0, 'collection', coll); d.insert(0, 'gene_list', lname); d.insert(0, 'background', bg_name)
            ORA_ALL.append(d)
ORA_ALL = pd.concat(ORA_ALL, ignore_index=True)
ORA_ALL.to_csv(os.path.join(RES, 'ora_08.csv'), index=False)

ora_summary = (ORA_ALL.groupby(['background', 'gene_list', 'collection'])
               .agg(sets_tested=('p', 'size'), min_p=('p', 'min'), min_fdr=('fdr_bh', 'min'),
                    n_fdr_lt_005=('fdr_bh', lambda s: int((s < 0.05).sum()))).reset_index())
display(ora_summary)
prim = ORA_ALL[(ORA_ALL.background == 'primary_top5000') & (ORA_ALL.overlap >= 2)]
print('\nTop sets with >= 2 overlapping genes (primary background), by p-value:')
display(prim.sort_values('p').head(15)[['gene_list', 'collection', 'gene_set', 'overlap', 'set_size_in_bg',
                                         'fold_enrichment', 'p', 'fdr_bh', 'overlap_genes']])

## 8b. Skorla korelasyona dayalı gen seti testi

Soru: Skor (LP) hangi yolaklarla birlikte hareket ediyor? Eğitimdeki 373 hastada her genin LP ile Spearman korelasyonu hesaplanır. Bir setin istatistiği = setteki ortalama ρ − evrendeki ortalama ρ (rekabetçi test).

- Evren: 5.000 genlik birincil arka plan **eksi modeldeki 44 gen**. Model genleri çıkarılmasaydı, onları içeren setler LP ile tanım gereği ilişkili çıkardı (döngüsellik).
- Sıfır dağılımı: LP'nin hastalar arasında 2.000 kez karıştırılması. Bu, genler arası korelasyonu korur (gen permütasyonunun bilinen yanlış pozitif sorununu önler).
- İkinci test (PAGE tipi, Kim ve Volsky 2005, *BMC Bioinformatics* 6:144): setin ortalama ρ'su aynı boyuttaki rastgele gen setlerine göre uç mu? Bu, gen seçiminden gelen değişkenliği hesaba katar ama genler arası korelasyonu yok sayar; permütasyon testi ise tersini yapar. Skor çok sayıda genle korele olduğunda permütasyon testi tek başına rastgele setleri bile "anlamlı" bulabilir (sentetik testte gördük). Bu yüzden **"sağlam" set = iki testte de FDR < 0,05**; yorum yalnızca bunlarla yapılır.
- Sağkalım kullanılmaz; bu, skorun hangi biyolojiyi izlediğinin betimlenmesidir, prognoz iddiası değildir.
- LP eğitim verisinde hesaplandığı için görünürdür (apparent); ama sağkalımla ilgili hiçbir tahmin yapılmadığından bu iyimserlik burada sorun değildir.

In [ ]:
UNIVERSE = [g for g in BG5000 if g not in model_set]
Ui = np.array([gpos[g] for g in UNIVERSE])
n = A.shape[0]
R = stats.rankdata(A[:, Ui], axis=0)
R = (R - R.mean(axis=0)) / R.std(axis=0, ddof=1)
rl = stats.rankdata(LP); rl = (rl - rl.mean()) / rl.std(ddof=1)
rho = R.T @ rl / (n - 1)
B = CONFIG['rank_test']['n_perm']
rng = np.random.default_rng(SEED + 1)
RL = np.stack([rng.permutation(rl) for _ in range(B)], axis=1)
rho_null = (R.T @ RL) / (n - 1)                       # genes x permutations
upos = {g: i for i, g in enumerate(UNIVERSE)}
pd.DataFrame({'entrez': UNIVERSE, 'symbol': [SYM.get(g, g) for g in UNIVERSE], 'spearman_rho_with_lp': rho}) \
  .sort_values('spearman_rho_with_lp').to_csv(os.path.join(RES, 'gene_lp_correlation_08.csv'), index=False)
print(f'Universe: {len(UNIVERSE)} genes | rho range {rho.min():.3f} to {rho.max():.3f} | '
      f'genes with |rho| > 0.3: {int((np.abs(rho) > 0.3).sum())}')

def rank_test(gsets, lo=10, hi=500):
    names, members = [], []
    for name, gs in gsets.items():
        idx = [upos[g] for g in gs if g in upos]
        if lo <= len(idx) <= hi:
            names.append(name); members.append(idx)
    rows_i = np.repeat(np.arange(len(names)), [len(m) for m in members])
    cols_i = np.concatenate(members)
    vals = np.concatenate([np.full(len(m), 1.0 / len(m)) for m in members])
    M = sparse.csr_matrix((vals, (rows_i, cols_i)), shape=(len(names), len(UNIVERSE)))
    obs = M @ rho - rho.mean()
    null = M @ rho_null - rho_null.mean(axis=0)
    p = (1 + (np.abs(null) >= np.abs(obs)[:, None]).sum(axis=1)) / (B + 1)
    z = obs / null.std(axis=1, ddof=1)
    lead = []
    for o, m in zip(obs, members):
        mm = np.array(m)
        top = mm[np.argsort(-np.sign(o) * rho[mm])[:6]]
        lead.append(';'.join(f'{SYM.get(UNIVERSE[i], UNIVERSE[i])}({rho[i]:+.2f})' for i in top))
    df = pd.DataFrame({'gene_set': names, 'n_genes': [len(m) for m in members],
                       'mean_rho_in_set': [rho[m].mean() for m in members], 'stat_vs_universe': obs,
                       'z_vs_null': z, 'direction': np.where(obs > 0, 'higher with higher risk score', 'higher with lower risk score'),
                       'p_perm': p, 'leading_genes': lead})
    df['fdr_bh'] = bh(df['p_perm'].to_numpy())
    # PAGE-type gene-sampling test (Kim & Volsky 2005): is the set mean extreme among random gene sets of the same size?
    k = df['n_genes'].to_numpy(float); Nu = len(UNIVERSE)
    se = rho.std() / np.sqrt(k) * np.sqrt((Nu - k) / (Nu - 1))
    df['z_gene_sampling'] = obs / se
    df['p_gene_sampling'] = 2 * stats.norm.sf(np.abs(df['z_gene_sampling']))
    df['fdr_gene_sampling'] = bh(df['p_gene_sampling'].to_numpy())
    df['robust'] = (df['fdr_bh'] < 0.05) & (df['fdr_gene_sampling'] < 0.05)
    return df.sort_values(['robust', 'z_gene_sampling'], ascending=[False, False],
                          key=lambda s: s if s.name == 'robust' else s.abs()).reset_index(drop=True)

RANK = {}
for coll in CONFIG['rank_test']['collections']:
    RANK[coll] = rank_test(GENESETS[coll])
    RANK[coll].insert(0, 'collection', coll)
    d = RANK[coll]
    print(f"{coll:9s}: {len(d):5d} sets | FDR<0.05 permutation {int((d.fdr_bh < 0.05).sum()):4d} | "
          f"gene-sampling {int((d.fdr_gene_sampling < 0.05).sum()):4d} | robust (both) {int(d.robust.sum()):4d} "
          f"(up with risk {int((d.robust & (d.stat_vs_universe > 0)).sum())}, down {int((d.robust & (d.stat_vs_universe < 0)).sum())})")
rank_all = pd.concat(RANK.values(), ignore_index=True)
rank_all.to_csv(os.path.join(RES, 'rank_test_08.csv'), index=False)
min_p = 1 / (B + 1)
print(f'(smallest attainable permutation p = {min_p:.5f})')
for coll in RANK:
    print(f'\nTop 10 - {coll}')
    display(RANK[coll].head(10)[['gene_set', 'n_genes', 'mean_rho_in_set', 'z_gene_sampling', 'direction', 'fdr_bh',
                                 'fdr_gene_sampling', 'robust', 'leading_genes']])

h = RANK['HALLMARK'].sort_values('z_gene_sampling')
fig, ax = plt.subplots(figsize=(7, 11))
ax.barh(range(len(h)), h['z_gene_sampling'], color=['#b2182b' if v > 0 else '#2166ac' for v in h['z_gene_sampling']])
for i, rb in enumerate(h['robust']):
    if rb:
        ax.text(h['z_gene_sampling'].iloc[i], i, ' *', va='center', fontsize=8)
ax.set_yticks(range(len(h))); ax.set_yticklabels([s.replace('HALLMARK_', '') for s in h['gene_set']], fontsize=7)
ax.axvline(0, color='black', lw=0.6)
ax.set_xlabel('Gene-sampling z (set mean rho vs random gene sets)')
ax.set_title('Hallmark sets vs risk score (training, model genes removed)\nred = higher with higher score; * FDR < 0.05 in both tests', fontsize=9)
save_fig(fig, 'hallmark_rank_test_08.png')

## 9. Bilinen DLBCL imzalarıyla ilişki (yalnızca eğitim)

Her imza için skor = imza genlerinin z-skorlarının ortalaması (13.747 ortak gen içinde bulunanlar). LP ile Spearman ρ, hasta bootstrap'ıyla %95 GA (1.000 örneklem). Model genleri imzayla örtüşüyorsa ρ ayrıca model genleri çıkarılarak verilir.

- **Lymph2Cx (COO):** Scott ve ark. 2014 (Blood). 8 ABC geni (TNFRSF13B, LIMD1, IRF4, CREB3L2, PIM2, CYB5R2, RAB7L1/RAB29, CCDC50) ve 7 GCB geni (MME, SERPINA9, ASB13, MAML3, ITPKB, MYBL1, S1PR2). COO skoru = ABC ortalaması − GCB ortalaması. Kontrol: bu skor yazarların ABC/GCB etiketlerini ayırmalı (3. adımda AUC 0,952).
- **Proliferasyon:** Hallmark E2F hedefleri, G2M kontrol noktası, MYC hedefleri V1.
- **Stroma:** Lenz ve ark. 2008 (NEJM) stromal-1 (hücre dışı matriks, histiyosit) ve stromal-2 (damar yoğunluğu) imzalarının gen listeleri burada makine tarafından okunabilir biçimde yok. **Vekil** olarak Hallmark EMT (ECM) ve Hallmark anjiyogenez kullanılır; bunlar Lenz imzalarının kendisi değildir.
- **Bağışıklık / NF-κB:** Hallmark IFN-γ yanıtı, inflamatuvar yanıt, TNFα/NF-κB, IL6/JAK/STAT3.
- **MSigDB C2 CGP içindeki DLBCL/lenfoma imzaları:** adı önceden belirlenmiş bir desene uyan bütün setler (seçim sonuç görmeden, yalnızca ada göre); ayrı BH grubu.

Okuma notu: rastgele gen setleri bile, skorla korele modülleri tesadüfen içerdiklerinde küçük korelasyon verebilir (sentetik testte |ρ| ≈ 0,1–0,15 görüldü). Bu yüzden |ρ| < 0,2 olan ilişkileri, FDR'si küçük olsa bile, zayıf say; yorumu ρ'nun büyüklüğü ve güven aralığı üzerinden yap.

Ek betimlemeler: LP'nin COO gruplarına göre dağılımı (Kruskal–Wallis) ve IPI ile Spearman korelasyonu (IPI'si bilinen hastalar). Son olarak LP'nin ne kadarının bilinen imzalarla açıklandığı: LP ~ imza skorları (model genleri çıkarılmış) doğrusal regresyon R² — betimleyici.

In [ ]:
Z = (A - A.mean(axis=0)) / np.where(A.std(axis=0, ddof=1) > 0, A.std(axis=0, ddof=1), 1.0)
L2CX_ABC = ['TNFRSF13B', 'LIMD1', 'IRF4', 'CREB3L2', 'PIM2', 'CYB5R2', 'RAB7L1', 'CCDC50']
L2CX_GCB = ['MME', 'SERPINA9', 'ASB13', 'MAML3', 'ITPKB', 'MYBL1', 'S1PR2']
abc_e = {s: entrez_for(s) for s in L2CX_ABC}; gcb_e = {s: entrez_for(s) for s in L2CX_GCB}
missing = [s for s, e in {**abc_e, **gcb_e}.items() if e is None]
print(f'Lymph2Cx genes found: {15 - len(missing)}/15', ('missing: ' + ', '.join(missing)) if missing else '')
check('at least 12 of 15 Lymph2Cx genes found', len(missing) <= 3, missing)

def zmean(entrez_list, drop_model=False):
    idx = [gpos[e] for e in entrez_list if e is not None and e in gpos and not (drop_model and e in model_set)]
    return (Z[:, idx].mean(axis=1) if idx else np.full(n, np.nan)), len(idx)

def l2cx(drop_model=False):
    a, na = zmean(list(abc_e.values()), drop_model); g, ng = zmean(list(gcb_e.values()), drop_model)
    return a - g, na + ng
coo_score, _ = l2cx()
lab = clin['coo'].to_numpy()
m_abc, m_gcb = lab == 'ABC', lab == 'GCB'
U = stats.mannwhitneyu(coo_score[m_abc], coo_score[m_gcb]).statistic
auc_l2cx = U / (m_abc.sum() * m_gcb.sum())
print(f'Lymph2Cx COO score separates the authors\' ABC vs GCB labels: AUC = {auc_l2cx:.3f} (step 3: 0.952 on 380 samples)')
if auc_l2cx < 0.85:
    print('WARNING: unexpectedly low AUC - check symbol mapping before interpreting COO results.')

SIG = {}   # name -> (group, entrez list, note)
SIG['LYMPH2CX_COO_SCORE'] = ('curated', None, 'Scott 2014; ABC minus GCB')
for s in CONFIG['signatures']['curated_hallmark_panel']:
    if s in GENESETS['HALLMARK']:
        SIG[s] = ('curated', sorted(GENESETS['HALLMARK'][s]), CONFIG['signatures']['proxies'].get(s, ''))
    else:
        print('note: Hallmark set not found:', s)
rx = re.compile(CONFIG['signatures']['c2_cgp_regex'])
cgp_sel = sorted(k for k in GENESETS['C2_CGP'] if rx.search(k))
print(f'C2 CGP sets matching the pre-specified name pattern: {len(cgp_sel)}')
for s in cgp_sel:
    SIG[s] = ('c2_cgp', sorted(GENESETS['C2_CGP'][s]), '')

scores, meta = {}, []
for name, (grp, ents, note) in SIG.items():
    if name == 'LYMPH2CX_COO_SCORE':
        sc, nf = l2cx(); sx, nfx = l2cx(True)
        ov = sum(e in model_set for e in list(abc_e.values()) + list(gcb_e.values()) if e)
    else:
        sc, nf = zmean(ents); sx, nfx = zmean(ents, True)
        ov = sum(e in model_set for e in ents)
    if nf < CONFIG['signatures']['min_genes_found'] and name != 'LYMPH2CX_COO_SCORE':
        continue
    scores[name] = sc; scores[name + '__nomodel'] = sx if nfx >= 5 else np.full(n, np.nan)
    meta.append({'signature': name, 'group': grp, 'n_genes_found': nf, 'n_model_genes_in_signature': ov, 'note': note})
meta = pd.DataFrame(meta)
S = pd.DataFrame(scores, index=primary)

def spearman_cols(mat, y):
    rm = stats.rankdata(mat, axis=0); ry = stats.rankdata(y)
    rm = (rm - rm.mean(0)) / rm.std(0, ddof=1); ry = (ry - ry.mean()) / ry.std(ddof=1)
    return rm.T @ ry / (len(y) - 1)

names = list(meta['signature'])
Smat = S[names].to_numpy()
rho_sig = spearman_cols(Smat, LP)
p_sig = np.array([stats.spearmanr(S[nm], LP).pvalue for nm in names])
Sx = S[[nm + '__nomodel' for nm in names]]
rho_x = np.array([stats.spearmanr(Sx[c], LP, nan_policy='omit').statistic if Sx[c].notna().all() else np.nan for c in Sx])
rng_b = np.random.default_rng(SEED + 2)
boot = np.empty((1000, len(names)))
for b in range(1000):
    ii = rng_b.integers(0, n, n)
    boot[b] = spearman_cols(Smat[ii], LP[ii])
meta['rho_with_lp'] = rho_sig
meta['ci95_low'], meta['ci95_high'] = np.percentile(boot, 2.5, axis=0), np.percentile(boot, 97.5, axis=0)
meta['p'] = p_sig
meta['rho_model_genes_removed'] = rho_x
meta['fdr_bh'] = np.nan
for grp in meta['group'].unique():
    mk = meta['group'] == grp
    meta.loc[mk, 'fdr_bh'] = bh(meta.loc[mk, 'p'].to_numpy())
meta = meta.sort_values(['group', 'rho_with_lp'], key=lambda s: s if s.name == 'group' else -s.abs()).reset_index(drop=True)
meta.to_csv(os.path.join(RES, 'signature_associations_08.csv'), index=False)
display(meta[meta.group == 'curated'][['signature', 'n_genes_found', 'n_model_genes_in_signature', 'rho_with_lp',
                                       'ci95_low', 'ci95_high', 'fdr_bh', 'rho_model_genes_removed', 'note']])
print('\nC2 CGP DLBCL/lymphoma signatures, top 15 by |rho|:')
display(meta[meta.group == 'c2_cgp'].head(15)[['signature', 'n_genes_found', 'n_model_genes_in_signature', 'rho_with_lp',
                                               'ci95_low', 'ci95_high', 'fdr_bh', 'rho_model_genes_removed']])

# --- clinical descriptives (no survival data)
groups = ['GCB', 'UNC', 'MHG', 'ABC']
coo_lp = pd.DataFrame([{'coo': g, 'n': int((lab == g).sum()), 'median_lp_per_sd': float(np.median(LPs[lab == g])),
                        'iqr_low': float(np.percentile(LPs[lab == g], 25)), 'iqr_high': float(np.percentile(LPs[lab == g], 75))}
                       for g in groups if (lab == g).any()])
kw = stats.kruskal(*[LPs[lab == g] for g in groups if (lab == g).any()])
ipi_ok = clin['ipi'].notna().to_numpy()
ipi_rho = stats.spearmanr(LP[ipi_ok], clin['ipi'].to_numpy()[ipi_ok])
print(f'\nLP by COO group (Kruskal-Wallis p = {kw.pvalue:.2e}):'); display(coo_lp)
print(f'Spearman(LP, IPI) among {int(ipi_ok.sum())} patients with IPI: rho = {ipi_rho.statistic:.3f}, p = {ipi_rho.pvalue:.2e}')

# --- how much of the score do known signatures explain? (descriptive OLS, model genes removed)
REG = ['LYMPH2CX_COO_SCORE', 'HALLMARK_E2F_TARGETS', 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION', 'HALLMARK_ANGIOGENESIS',
       'HALLMARK_INTERFERON_GAMMA_RESPONSE', 'HALLMARK_TNFA_SIGNALING_VIA_NFKB']
REG = [r for r in REG if r + '__nomodel' in S and S[r + '__nomodel'].notna().all()]
def ols_r2(cols):
    Xr = S[[c + '__nomodel' for c in cols]].to_numpy()
    Xr = (Xr - Xr.mean(0)) / Xr.std(0, ddof=1)
    D = np.column_stack([np.ones(n), Xr])
    beta, *_ = np.linalg.lstsq(D, LPs, rcond=None)
    resid = LPs - D @ beta
    r2 = 1 - resid.var() / LPs.var()
    adj = 1 - (1 - r2) * (n - 1) / (n - len(cols) - 1)
    return r2, adj, beta[1:]
r2_coo, adj_coo, _ = ols_r2(['LYMPH2CX_COO_SCORE'])
r2_all, adj_all, betas = ols_r2(REG)
reg_tab = pd.DataFrame({'predictor': REG, 'standardized_beta': betas})
print(f'\nLP explained by Lymph2Cx COO score alone: R^2 = {r2_coo:.3f}')
print(f'LP explained by {len(REG)} signature scores together: R^2 = {r2_all:.3f} (adjusted {adj_all:.3f}); '
      f'unexplained share {1 - r2_all:.2f}')
display(reg_tab)

cur = meta[meta.group == 'curated'].iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={'width_ratios': [1.25, 1]})
ax = axes[0]
yy = np.arange(len(cur))
ax.errorbar(cur['rho_with_lp'], yy, xerr=[cur['rho_with_lp'] - cur['ci95_low'], cur['ci95_high'] - cur['rho_with_lp']],
            fmt='o', color='black', ms=4, capsize=2)
ax.scatter(cur['rho_model_genes_removed'], yy, marker='x', color='#d6604d', label='model genes removed', zorder=3)
ax.set_yticks(yy); ax.set_yticklabels([s.replace('HALLMARK_', 'H_') for s in cur['signature']], fontsize=7)
ax.axvline(0, color='grey', lw=0.6); ax.set_xlabel('Spearman rho with risk score (95% bootstrap CI)')
ax.legend(fontsize=7); ax.set_title('Known signatures vs 44-gene score (training)', fontsize=9)
ax = axes[1]
for g, c in zip(['GCB', 'UNC', 'MHG', 'ABC'], ['#4393c3', '#999999', '#5aae61', '#d6604d']):
    mk = lab == g
    ax.scatter(coo_score[mk], LPs[mk], s=8, alpha=0.6, color=c, label=f'{g} (n={mk.sum()})')
ax.set_xlabel('Lymph2Cx COO score (ABC - GCB)'); ax.set_ylabel('Risk score (LP, SD units)')
ax.legend(fontsize=7, markerscale=1.5); ax.set_title('Score vs cell of origin (training)', fontsize=9)
save_fig(fig, 'signatures_vs_score_08.png')

## 10. Literatür notları (yalnızca doğrulanmış kaynaklar)

Aşağıdaki işlev bilgileri birincil makalelerden ya da UniProt/NCBI kayıtlarından alınmıştır. **Bu genlerin DLBCL'deki rolüne dair doğrudan bir kanıt bulunmadıkça öyle bir iddia kurulmaz.** Katsayı yönleri 4b'den bilinenlerdir; AGBL4 ve diğerlerinin yönü yukarıdaki tablodadır.

| Gen (4b bootstrap) | Modeldeki yön | Bilinen işlev | Kaynak | DLBCL bağlantısı |
|---|---|---|---|---|
| PXDNL (%84) | negatif (yüksek ifade, düşük risk) | Peroksidaz–siklooksijenaz ailesi; PXDN'nin aksine işlevi pratikte bilinmiyor; insanda kalpte, kardiyomiyositlerde eksprese | Péterfi ve ark. 2014, *Cardiovasc Res* 101(3):342; Kovács ve ark. 2021, *Antioxidants* 10:1565 | Bilinen doğrudan bağlantı yok. Lenfoma dokusunda hangi hücreden geldiği belirsiz; DASL prob artefaktı olasılığı dışlanamaz |
| JAML / AMICA1 (%68) | negatif | Bağlantı molekülü benzeri protein; CAR (CXADR) ile bağlanıp PI3K üzerinden γδ T hücrelerine eş-uyarı verir; CD8 doku-yerleşik bellek T hücrelerinde yüksek; farelerde antitümör γδ/CD8 TIL yanıtını destekler | Witherden ve ark. 2010, *Science* 329:1205; Verdino ve ark. 2010, *Science* 329:1210; McGraw ve ark. 2021, *J Exp Med* 218:e20202644; Eschweiler ve ark. 2023, *Cell Rep* 42:112040 | DLBCL'de doğrudan çalışma bulunamadı. Yön, "tümörü infiltre eden etkin T hücresi = daha iyi seyir" fikriyle uyumlu, ama bu bir hipotezdir; JAML nötrofil ve monositlerde de var |
| PLAC8L1 (%60) | pozitif (yüksek ifade, yüksek risk) | Pratikte tanımlanmamış; PLAC8 motifli protein ailesi; NCBI'da işlev özeti yok | NCBI Gene 153770; PLAC8 ailesi için Mao ve ark. 2021, *Biomark Res* 9:73 | Bilinen bağlantı yok; yorum yapılamaz |
| AGBL4 / CCP6 (%50) | tabloya bakınız | Sitozolik karboksipeptidaz 6; tübülin ve diğer proteinlerden uzun poliglutamat zincirlerini kısaltır; cGAS'ın deglutamilasyonu ile antiviral yanıtı düzenler | Rogowski ve ark. 2010, *Cell*; Xia ve ark. 2016, *Nat Immunol* (doi 10.1038/ni.3397); UniProt Q5VU57 | Bilinen doğrudan bağlantı yok |
| CXCL2 (%44, kararsız) | pozitif | Nötrofil kemokini (CXCR2 ligandı) | — (genel bilgi) | DLBCL'de tümörle ilişkili nötrofil imzası R-CHOP sonrası kötü seyirle ilişkilendirilmiş (Manfroi ve ark. 2018, *Blood Cancer J*, doi 10.1038/s41408-018-0099-y). CXCL2 için tek başına kanıt değil |

Yolakların literatür yorumu bu not defterinin **sonuçlarına bağlıdır** (hangi Hallmark/Reactome setlerinin öne çıktığı ancak çalıştırmadan sonra bilinir). Sonuçları paylaştığında öne çıkan yolakları kaynaklarla yorumlayıp ilerleme özetine ekleyeceğiz.

Genel kaynaklar: Lenz ve ark. 2008, *N Engl J Med* 359:2313 (stromal imzalar); Scott ve ark. 2014, *Blood* (Lymph2Cx); Subramanian ve ark. 2005, *PNAS* (GSEA/MSigDB); Liberzon ve ark. 2015, *Cell Systems* (Hallmark); Benjamini ve Hochberg 1995, *J R Stat Soc B* (FDR).

In [ ]:
LITERATURE = [
    {'gene': 'PXDNL', 'claim': 'peroxidasin-like protein; function practically unknown; expressed in human cardiomyocytes',
     'source': 'Peterfi et al. 2014 Cardiovasc Res 101(3):342; Kovacs et al. 2021 Antioxidants 10:1565 (doi 10.3390/antiox10101565)',
     'dlbcl_link': 'none found'},
    {'gene': 'JAML', 'claim': 'binds CXADR (CAR); costimulates gamma-delta T cells via PI3K; enriched in CD8 tissue-resident memory TILs; supports antitumor CD8/gamma-delta TIL responses in mice',
     'source': 'Witherden et al. 2010 Science 329:1205; Verdino et al. 2010 Science 329:1210; McGraw et al. 2021 J Exp Med 218:e20202644; Eschweiler et al. 2023 Cell Rep 42:112040',
     'dlbcl_link': 'none found (direction consistent with an infiltrating-T-cell hypothesis; untested)'},
    {'gene': 'PLAC8L1', 'claim': 'uncharacterized PLAC8-motif protein', 'source': 'NCBI Gene 153770', 'dlbcl_link': 'none found'},
    {'gene': 'AGBL4', 'claim': 'cytosolic carboxypeptidase 6 (CCP6); long-chain protein deglutamylase; deglutamylates cGAS',
     'source': 'Rogowski et al. 2010 Cell; Xia et al. 2016 Nat Immunol (doi 10.1038/ni.3397); UniProt Q5VU57', 'dlbcl_link': 'none found'},
    {'gene': 'CXCL2', 'claim': 'neutrophil chemokine (CXCR2 ligand)',
     'source': 'tumor-associated neutrophils and poor prognosis in DLBCL: Manfroi et al. 2018 Blood Cancer J (doi 10.1038/s41408-018-0099-y)',
     'dlbcl_link': 'indirect (neutrophil signature), not CXCL2 itself'},
]
pd.DataFrame(LITERATURE).to_csv(os.path.join(RES, 'literature_genes_08.csv'), index=False)
print('Literature table saved (gene functions only; pathway literature is added after results are reviewed).')

## 11. Özet, manifesto ve çalıştırma kaydı

Sayısal özet `results_08.json` ve `summary_08.txt` dosyalarına yazılır. Manifesto her çıktının SHA-256'sını, okunan veri dosyalarını, GMT özetlerini ve ortamı kaydeder. Her çalıştırma `run_log_08.jsonl` dosyasına bir satır ekler (yeniden çalıştırmalar görünür kalır).

In [ ]:
def top_sets(df, k=5):
    d = df[df.robust]
    return [{'gene_set': r.gene_set, 'z_gene_sampling': round(float(r.z_gene_sampling), 2), 'fdr_perm': float(r.fdr_bh),
             'fdr_gene_sampling': float(r.fdr_gene_sampling), 'direction': r.direction} for r in d.head(k).itertuples()]

stable = gene_tab[gene_tab.stable_ge_50pct]
results = {
    'code_version': CODE_VERSION, 'test_mode': TEST_MODE, 'exploratory': True,
    'population': {'n': len(primary), 'qc_excluded': len(excluded), 'coo_counts': coo_counts},
    'model': {'n_genes': len(genes), 'risk_up': n_up, 'risk_down': n_down,
              'stable_genes_ge_50pct': [{'symbol': r.symbol_GPL570_current, 'freq': float(r.bootstrap_freq), 'coef': float(r.coef)}
                                        for r in stable.itertuples()],
              'symbols_differ_between_platforms': int(gene_tab.symbol_differs.sum())},
    'ora': {'n_fdr_lt_005_primary_background': int(((ORA_ALL.background == 'primary_top5000') & (ORA_ALL.fdr_bh < 0.05)).sum()),
            'n_fdr_lt_005_sensitivity_background': int(((ORA_ALL.background == 'sensitivity_all13747') & (ORA_ALL.fdr_bh < 0.05)).sum())},
    'rank_test': {c: {'sets_tested': int(len(RANK[c])), 'n_fdr_perm_lt_005': int((RANK[c].fdr_bh < 0.05).sum()),
                      'n_fdr_gene_sampling_lt_005': int((RANK[c].fdr_gene_sampling < 0.05).sum()),
                      'n_robust': int(RANK[c].robust.sum()), 'top_robust_sets': top_sets(RANK[c])} for c in RANK},
    'signatures': {'lymph2cx_auc_abc_vs_gcb': float(auc_l2cx),
                   'curated': [{'signature': r.signature, 'rho': round(float(r.rho_with_lp), 3),
                                'ci95': [round(float(r.ci95_low), 3), round(float(r.ci95_high), 3)], 'fdr': float(r.fdr_bh),
                                'rho_model_genes_removed': None if np.isnan(r.rho_model_genes_removed) else round(float(r.rho_model_genes_removed), 3)}
                               for r in meta[meta.group == 'curated'].itertuples()],
                   'c2_cgp_tested': int((meta.group == 'c2_cgp').sum()),
                   'c2_cgp_fdr_lt_005': int(((meta.group == 'c2_cgp') & (meta.fdr_bh < 0.05)).sum())},
    'clinical_descriptive': {'kruskal_p_lp_by_coo': float(kw.pvalue), 'coo_median_lp_per_sd': coo_lp.set_index('coo')['median_lp_per_sd'].round(3).to_dict(),
                             'spearman_lp_ipi': float(ipi_rho.statistic), 'n_ipi': int(ipi_ok.sum())},
    'variance_explained': {'r2_lymph2cx_only': float(r2_coo), 'r2_signatures': float(r2_all), 'adj_r2_signatures': float(adj_all),
                           'predictors': REG},
    'caveats': ['Exploratory analysis; no hypothesis test; p-values and FDRs are descriptive.',
                'Gene selection was unstable in 4b (4/44 genes selected in >= 50% of bootstraps); interpret the score as a whole.',
                'Training set only; LP is apparent (in-sample). No survival data used.',
                'Stromal signatures are Hallmark proxies, not the Lenz 2008 gene lists.',
                'COO labels come from the authors\' classifier on the same arrays.'],
}
with open(os.path.join(RES, 'results_08.json'), 'w') as f:
    json.dump(results, f, indent=2, sort_keys=True, default=float)

lines = [f'Step 8 - exploratory biological interpretation (code {CODE_VERSION}){"  [TEST MODE]" if TEST_MODE else ""}',
         f'Population: GSE181063 training, primary n = {len(primary)}; survival data not read; test/external expression not opened.',
         f'Model: 44 genes ({n_up} risk-up, {n_down} risk-down); stable (>= 50% bootstrap): ' +
         ', '.join(f'{r.symbol_GPL570_current} {r.bootstrap_freq:.0%}' for r in stable.itertuples()),
         f'ORA (background = 5,000-gene selection pool): sets with FDR < 0.05: {results["ora"]["n_fdr_lt_005_primary_background"]}'
         f' (13,747-gene background: {results["ora"]["n_fdr_lt_005_sensitivity_background"]})']
for c in RANK:
    r_ = results['rank_test'][c]
    lines.append(f'Rank test {c}: robust {r_["n_robust"]}/{r_["sets_tested"]} (FDR < 0.05 in both permutation and gene-sampling tests); top: ' +
                 ('; '.join(f'{t["gene_set"]} (z {t["z_gene_sampling"]:+.1f})' for t in r_['top_robust_sets'][:3]) or 'none'))
lines.append(f'Lymph2Cx AUC (ABC vs GCB labels) {auc_l2cx:.3f}; rho(LP, COO score) {meta.loc[meta.signature == "LYMPH2CX_COO_SCORE", "rho_with_lp"].iloc[0]:+.3f}')
lines.append(f'R^2 of LP: Lymph2Cx alone {r2_coo:.3f}; six signature scores {r2_all:.3f}')
lines.append(f'LP by COO Kruskal-Wallis p = {kw.pvalue:.2e}; Spearman(LP, IPI) = {ipi_rho.statistic:+.3f} (n = {int(ipi_ok.sum())})')
lines += ['CAVEATS: ' + c for c in results['caveats']]
summary = '\n'.join(lines)
open(os.path.join(RES, 'summary_08.txt'), 'w').write(summary + '\n')
pd.DataFrame(CHECKS).to_csv(os.path.join(RES, 'checks_08.csv'), index=False)

outputs = {}
for d in (RES, FIG):
    for fn in sorted(os.listdir(d)):
        p = os.path.join(d, fn)
        if os.path.isfile(p) and fn not in ('manifest_08.json', 'run_log_08.jsonl'):
            outputs[os.path.relpath(p, PROJECT)] = sha256(p)
manifest = {
    'step': '08_biological_interpretation', 'code_version': CODE_VERSION, 'test_mode': TEST_MODE,
    'run_started_utc': RUN_STARTED, 'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'plan_lock_sha256': lock_sha, 'frozen_model_sha256': model_sha,
    'verification': {k: {kk: v[kk] for kk in ('records', 'verified', 'unresolved', 'skipped_large')} |
                     {'notebook_mismatch': v['notebook_mismatch'], 'n_hashes': len(HEXES[k])} for k, v in verification.items()},
    'checks': CHECKS,
    'data_read': {p: sha256(P(p)) for p in DATA_READ},
    'data_not_opened': ['test-set expression', 'external-cohort expression', 'all survival columns'],
    'analysis_config': {'file': os.path.relpath(cfg_path, PROJECT), 'sha256': sha256(cfg_path)},
    'gene_sets': GMT_RECORD, 'outputs': outputs,
    'environment': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                    'scipy': scipy.__version__, 'matplotlib': matplotlib.__version__},
}
with open(os.path.join(RES, 'manifest_08.json'), 'w') as f:
    json.dump(manifest, f, indent=2, sort_keys=True)
with open(os.path.join(RES, 'run_log_08.jsonl'), 'a') as f:
    f.write(json.dumps({'run_finished_utc': manifest['run_finished_utc'], 'code_version': CODE_VERSION, 'test_mode': TEST_MODE,
                        'results_sha256': outputs.get(os.path.relpath(os.path.join(RES, 'results_08.json'), PROJECT))}) + '\n')
print(summary)
print('\nmanifest_08.json SHA-256:', sha256(os.path.join(RES, 'manifest_08.json')))

## Sonraki adım

Çıktıları (özellikle `summary_08.txt`, `rank_test_08.csv` ilk satırları ve `signature_associations_08.csv`) paylaş. Öne çıkan yolakları kaynaklarla yorumlayıp Adım 8'in ilerleme özetini yazacağız; ardından Adım 9 (rapor, GitHub sayfası, etkileşimli sayfa).